# Exercise 3.3 — The best fit line

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day3/day3-02b-best-fit-line.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

Exercise 3.2 narrowed the choice of loss function down to the mean squared
error (MSE). This exercise uses the MSE to identify the **line of best fit**.
You will calculate the MSE of a line by hand, see that trying lines one at a
time gets you nowhere, and then look at the MSE of every candidate line at
once, as a surface in 3D. The lowest point of that surface is the best fit
line.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression

Load the Day 3 dataset. To understand the calculation properly, make the
dataset even smaller to begin with: the first three people only.

In [ ]:
glu = pd.read_csv("data/glucose_study.csv")

glu3 = glu.iloc[:3]
glu3

In [ ]:
age     = glu3["age"]       # x, the feature
glucose = glu3["glucose"]   # y, the outcome label
m       = len(age)          # the number of samples

### The MSE as a function of the two parameters

We have narrowed things down: we continue with the MSE loss function, to
identify the line of best fit. The MSE can also be written as

$$\text{MSE} = \frac{1}{m} \sum_{i=1}^{m}
\left( y_i - \left( \hat{\beta}_0 + \hat{\beta}_1 x_i \right) \right)^2$$

because the predicted value comes out of the model,
$\hat y_i = \hat\beta_0 + \hat\beta_1 x_i$. The two ways of writing the MSE
mean the same thing.

> **Pause for understanding!**
>
> Make sure you can say where every symbol comes from. $x_i$ and $y_i$ are the age
> and the glucose of person $i$, and $m$ is the number of people: these come from
> the data and do not change. $\hat\beta_0$ and $\hat\beta_1$ are the two
> parameters of the line: these are what we are free to choose.

This can be coded like this:

In [ ]:
def mse_practical(beta0, beta1, x, y, m):
    return (1 / m) * np.sum((y - (beta0 + beta1 * x)) ** 2)

Put it to the test with the parameter values $\hat\beta_0 = 3$ and
$\hat\beta_1 = 5$. Remember, we already have $y$ (the outcome label, glucose),
$x$ (the feature, age) and $m$ (the number of samples) defined.

In [ ]:
mse_practical(3, 5, age, glucose, m)

Remember this is the same as:

In [ ]:
mse_practical(beta0=3, beta1=5, x=age, y=glucose, m=m)

Now calculate it by hand, on a piece of paper! Use Python as a calculator,
because the numbers are big, but make sure you get the concept.

| Person | Age $x_i$ | Glucose $y_i$ | Predicted $3 + 5x_i$ | Error $y_i - \text{predicted}$ | Error squared |
|:-:|:-:|:-:|:-:|:-:|:-:|
| 1 | 28 | 194 | 143 | 51 | 2601 |
| 2 | 44 | 129 | 223 | −94 | 8836 |
| 3 | 26 | 132 | 133 | −1 | 1 |

$$\text{MSE} = \frac{2601 + 8836 + 1}{3} = 3812.67$$

In [ ]:
((194 - 143)**2 + (129 - 223)**2 + (132 - 133)**2) / 3

Same as above!

Now do the same for $\hat\beta_0 = 4$ and $\hat\beta_1 = 7$: first by hand,
then with the function.

In [ ]:
# Replace every ______ below, then run this cell.
mse_practical(beta0=______, beta1=______, x=age, y=glucose, m=m)

The MSE is lower with the parameters $\hat\beta_0 = 3$, $\hat\beta_1 = 5$. But
is it the lowest it can go? Let's plot!

This plot has the two lines with the parameters we just tried. The one with the
lower loss fits the data better.

In [ ]:
ages = np.array([0, 60])   # the two ends of each line

fig, ax = plt.subplots()
ax.scatter(glu3["age"], glu3["glucose"], color="black", s=60, zorder=3)
ax.plot(ages, 3 + 5 * ages, color="blue", label="B0 = 3, B1 = 5")
ax.plot(ages, 4 + 7 * ages, color="red", label="B0 = 4, B1 = 7")
ax.set_xlim(0, 60)
ax.set_ylim(0, 350)
ax.set_title("Different betas")
ax.set_xlabel("Age")
ax.set_ylabel("Glucose")
ax.legend()
plt.show()

### Trying many lines

We could try this out for a whole range of different parameters. But that is
very slow. For example, let's go back to more data points, all 80 people:

In [ ]:
age     = glu["age"]
glucose = glu["glucose"]
m       = len(age)

Here are twelve candidate lines: every combination of three intercepts (the
colours) and four slopes (the line types).

In [ ]:
ages = np.array([glu["age"].min(), glu["age"].max()])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(glu["age"], glu["glucose"], color="black", s=25, zorder=3)
colours = {100: "tab:red", 110: "tab:green", 120: "tab:blue"}    # one colour per intercept
styles = {0: "-", 0.25: "--", 0.5: "-.", 0.75: ":"}              # one line type per slope
for b0 in colours:
    for b1 in styles:
        ax.plot(ages, b0 + b1 * ages, color=colours[b0], linestyle=styles[b1],
                label=f"B0 = {b0}, B1 = {b1}")
ax.set_title("Effect of different B0 and B1 values on the regression line")
ax.set_xlabel("Age (feature)")
ax.set_ylabel("Glucose (label)")
ax.legend(title="Parameter combination", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

We cannot just keep trying at random! How does `LinearRegression()` find the
parameters that correspond to the lowest MSE?

We need a way to find the combination of parameters that gives the minimum
value of the MSE (the loss function we chose), and so the linear regression
model that best fits the data points. But what does the minimum of the loss
function look like?

### The loss function in 3D

Let's plot the function, to understand it better! Three quantities vary: the two
parameters $\hat\beta_0$ and $\hat\beta_1$, and the MSE they produce. To see all
three at once we need a 3D plot.

$$\text{MSE}(\hat{\beta}_0, \hat{\beta}_1) = \frac{1}{m} \sum_{i=1}^{m}
\left( y_i - \left( \hat{\beta}_0 + \hat{\beta}_1 x_i \right) \right)^2$$

Take many values of each parameter, closely spaced:

In [ ]:
B0_values = np.arange(60, 161, 1)                   # 60, 61, ..., 160
B1_values = np.round(np.linspace(-1, 2, 151), 2)    # -1, -0.98, ..., 2

We can calculate the MSE for each combination, the same way as we did by hand
before, and store the results in a table:

In [ ]:
df = pd.DataFrame(
    [{"B0": b0, "B1": b1, "MSE": mse_practical(b0, b1, age, glucose, m)}
     for b0 in B0_values for b1 in B1_values]
)

df.head()

That is 101 intercepts times 151 slopes: 15,251 candidate lines, each with its
MSE. Now let's plot this function! Every point on the floor of the plot is one
candidate line, a pair of $\hat\beta_0$ and $\hat\beta_1$. The height of the
surface above it is the MSE of that line.

In [ ]:
from matplotlib.colors import LogNorm

# plot_surface() wants the MSE as a matrix: one row per B1, one column per B0
MSE_matrix = df.pivot(index="B1", columns="B0", values="MSE")
B0_grid, B1_grid = np.meshgrid(MSE_matrix.columns, MSE_matrix.index)
best = df[df["MSE"] == df["MSE"].min()]    # the lowest point, to mark it in red

fig = plt.figure(figsize=(7, 5.5))
ax = fig.add_subplot(projection="3d", computed_zorder=False)
ax.plot_surface(B0_grid, B1_grid, MSE_matrix.values, cmap="magma", norm=LogNorm())
ax.scatter(best["B0"], best["B1"], best["MSE"], color="red", s=70, zorder=5)
ax.view_init(elev=22, azim=-55)    # the angle we look from: try other numbers
ax.set_xlabel("B0 (intercept)")
ax.set_ylabel("B1 (slope)")
ax.set_zlabel("MSE", labelpad=12)
plt.show()

What is the minimum MSE? The red point! The surface is curved like a bowl, and
the idea to take away is that **the minimum is the single point at the bottom of
this 3D curved surface**. A surface with this shape is called **convex**:
wherever you stand on it, going downhill takes you to the same lowest point.

Here is the same surface again. This one you can turn: drag it with the mouse to
look at it from every side, and scroll to zoom. Hover over the surface to read
off the parameters and the MSE of any candidate line.

In [ ]:
# This plot needs the plotly package: pip install plotly
import plotly.graph_objects as go

fig_3d = go.Figure(go.Surface(x=MSE_matrix.columns, y=MSE_matrix.index,
                              z=MSE_matrix.values, colorbar={"title": "MSE"}))
fig_3d.add_trace(go.Scatter3d(x=best["B0"], y=best["B1"], z=best["MSE"],
                              mode="markers", marker={"size": 7, "color": "red"},
                              name="Minimum MSE"))
fig_3d.update_layout(scene={"xaxis_title": "B0 (intercept)",
                            "yaxis_title": "B1 (slope)",
                            "zaxis_title": "MSE",
                            "camera": {"eye": {"x": 1.5, "y": -1.7, "z": 0.9}}})
fig_3d.show()

We can read the minimum, and the parameters that produce it, out of the table:

In [ ]:
df["MSE"].min()

In [ ]:
# Find the minimum point
df[df["MSE"] == df["MSE"].min()]

This is the same surface once more, seen from directly above, like a map. Dark
is low and light is high, and the red point is the same minimum. Notice that
the low region is a long diagonal valley rather than a round hollow: a line with
a higher slope and a lower intercept fits these points almost as well.
Exercise 3.4 walks down this valley.

In [ ]:
fig, ax = plt.subplots()
cs = ax.contourf(MSE_matrix.columns, MSE_matrix.index, np.log10(MSE_matrix.values),
                 levels=20, cmap="magma")
ax.scatter(best["B0"], best["B1"], color="red", zorder=3)
cbar = fig.colorbar(cs, ax=ax, label="log10(MSE)")
ax.set_xlabel("B0 (intercept)")
ax.set_ylabel("B1 (slope)")
plt.show()

These parameters are very similar to what we obtain from `LinearRegression()`.
They would be the same if we had sampled the MSE finely enough to reach those
decimal places.

In [ ]:
fit1 = LinearRegression()
fit1 = fit1.fit(glu[["age"]], glu["glucose"])

print(fit1.intercept_, fit1.coef_[0])

The table's best candidate is an intercept of 110 and a slope of 0.30, which is
the very line you scored in Exercise 3.2. The fitted model has 110.30 and 0.289.
Fitting a linear regression **is** finding the bottom of this surface.

Trying every combination only worked here because there are just two
parameters, and because we already knew roughly where to look. Exercise 3.4
(gradient descent) and Exercise 3.5 (ordinary least squares) are the two ways
of getting to the bottom without trying everything.

> **Learning points**
>
> - The MSE is a function of the two parameters: every pair of intercept and
>   slope is one candidate line, with one MSE.
> - Those MSE values form a bowl-shaped (convex) surface with a single lowest
>   point.
> - Fitting a linear regression means finding that lowest point. Trying every
>   combination only works when there are very few parameters.

## Stretch

> **Stretch — narrow the grid**
>
> Take the best candidate you found and build a second set of `B0_values` and
> `B1_values` centred on it, ten times more closely spaced. Calculate the MSE
> table again and find its minimum. How close are you now to the fitted model, to
> three decimal places? How many rounds of this would you need to match it
> exactly, and what does that tell you about the method?

## To explore further

[Ordinary least squares regression, explained visually](https://setosa.io/ev/ordinary-least-squares-regression/)
is a visual walk through least squares. Use it to understand what outliers do
to the fitted line, and what linear regression looks like in more dimensions.